# 251114 - Learning about Spark

## Pandas APIs on Spark
### Scaling pandas with Spark
Pandas DataFrames vs Spark DataFrames:
- Pandas DataFrames are mutable, eagerily evaluated and maintain row order. They are restricted to a single machine. They perform well with small datasets
- Spark DataFrames are distributed, lazily evaluated, inmutable, do not maintain row order. They perform well with big datasets
- Pandas APIs on Spark (aka Pyspark.pandas) is the way to use pandas syntax on Spark DataFrames. However, it is not as efficient as implemting the solution natively in Spark

In [0]:
import pandas as pd
# Remember how to read in CSV from pandas
file_path = "/Workspace/Users/almayo@gmail.com/datasets/ticket_details.csv"
df_pandas = pd.read_csv(file_path)
display(df_pandas.head())

# Pandas APIs on Spark
import pyspark.pandas as ps
# Task: Read CSV file
"""
OBS! A pre-requisite is to upload the CSV file to a Unity Catalog volume and use the volume path
First I had to create a volume called 'learning' under Catalog/workspace/default and then upload the CSV file to the volume
"""
file_path = "/Volumes/workspace/default/learning/ticket_details.csv"
df_pandas_on_spark = ps.read_csv(file_path, inferSchema=True, multiLine=True, escape='"')
display(df_pandas_on_spark.head())


#df_weather = spark.table('samples.accuweather.forecast_daily_calendar_imperial')
#display(df_weather)

## Converting from "pandas API on Spark" df to Spark df

In [0]:
df_spark = df_pandas_on_spark.to_spark()
display(df_spark)

## Converting from Spark df to "pandas API on Spark" df
This conversion might be useful to use some of the pandas methods and get, for example, visualizations or aggregations on Series

In [0]:
# Alt. 1:
df_pandas_on_spark2 = ps.DataFrame(df_spark)
# Alt. 2:
# df_pandas_on_spark2 = df_spark.pandas_api()
display(df_pandas_on_spark2.head())

## SQL on 'pandas API on Spark' df

In [0]:
import pyspark.pandas as ps
import pandas as pd

file_path = "/Workspace/Users/almayo@gmail.com/datasets/ticket_details.csv"
df_pandas = pd.read_csv(file_path)

df_pandas_on_spark = ps.from_pandas(df_pandas)
df_pandas_on_spark.to_table("df_pandas_view")

result = ps.sql("SELECT * FROM df_pandas_view")
display(result.head())

# 251115 PySpark Basics
https://docs.databricks.com/aws/en/pyspark/basics

## Import SQL functions and data types

In [0]:
# import select functions and types
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.functions import floor, round

# import modules using an alias
import pyspark.sql.types as T
import pyspark.sql.functions as F

## DDL: Create a DataFrame

### Create a DataFrame with specified values

In [0]:
"""
Method: spark.createDataFrame()
Args:
  - data: defined as a list of nested tuples
  - schema:
    - Alt. 1: 'Simple schema': list of column names. The data types are automatically infered.
    - Alt. 2: 'StructType': list of StructField objects
"""
# Alt.1
df_children = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = ['name', 'age'])
display(df_children)

# Alt.2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

df_children_with_schema = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = StructType([
    StructField('name', StringType(), True), # True = Nullable
    StructField('age', IntegerType(), True)
  ])
)
display(df_children_with_schema)

### Create a DataFrame from a table in Unity Catalog
For learning purposes, we may find several sample tables under Catalog/Delta Shares Received/samples/accuweather/<multiple_tables>

In [0]:
# Find the table path by righ click on the table to access the table details
df_weather = spark.table('samples.accuweather.forecast_daily_calendar_imperial')
display(df_weather) 

### Create a DataFrame from an uploaded file
As a pre-req, the file must be first uploaded to a Unity Catalog volume.
Steps:
1.- Create a volume from the Catalog Explorer menu under the schema (db icon) that you want
2.- Navigate to the new or existing volume
3.- Upload the file (e.g. CSV file) to the volume ("Upload to this volume" button)
4.- Copy volume path
5.- Call the spark.read() method

In [0]:
volume_file_path = "/Volumes/workspace/default/learning/ticket_details.csv"

df_csv = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load(volume_file_path)
)
display(df_csv)

### Create a DataFrame from a file

In [0]:
display(dbutils.fs.ls('/databricks-datasets/samples/'))
# Equivalent to %fs ls '/databricks-datasets' (run it in a solo cell)

In [0]:
df_population = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load("/databricks-datasets/samples/population-vs-price/data_geo.csv")
)
display(df_population)

### Create a DataFrame from a JSON response
Skipping this one for now

##DML: Transform data with DataFrames